In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import csv
import math
import pickle
import random

# **Read Inputs**

In [2]:
words = []

current_word_touchpoints = None
with open('data_2.txt', 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith('===Word'):
            if current_word_touchpoints is not None:
                words.append(current_word_touchpoints)
            current_word_touchpoints = []
        elif line:
            try:
                parts = line.split()
                if len(parts) == 3:
                    char = parts[0]
                    x = float(parts[1])
                    y = float(parts[2])
                    current_word_touchpoints.append((char, x, y))
            except (ValueError, IndexError):
                pass

if current_word_touchpoints:
    words.append(current_word_touchpoints)

words

[[('a', 4.9218874, 13.221646), ('t', 16.132853, 7.0692873)],
 [('f', 13.671909, 12.948208),
  ('i', 25.942448, 7.70731),
  ('r', 12.03128, 7.160434),
  ('s', 7.48537, 13.540659),
  ('t', 15.346718, 7.3427258)],
 [('y', 20.91802, 6.65913),
  ('o', 29.633863, 8.071894),
  ('u', 23.754942, 8.61877)],
 [('r', 12.03128, 7.6161637),
  ('u', 24.711975, 7.7984557),
  ('n', 24.575256, 18.189108)],
 [('r', 13.159212, 8.61877),
  ('i', 26.831121, 7.3427258),
  ('s', 7.3144712, 12.993782),
  ('k', 27.719795, 12.629196)],
 [('f', 14.458044, 11.5810175),
  ('a', 5.4345837, 12.857063),
  ('i', 25.49811, 7.70731),
  ('l', 31.035233, 12.629196),
  ('u', 23.720762, 7.0692873),
  ('r', 12.988314, 8.436479),
  ('e', 9.331078, 7.4794445)],
 [('w', 5.4687634, 8.254187),
  ('a', 4.5117297, 13.176072),
  ('t', 16.37211, 7.1148615),
  ('c', 14.936561, 17.414364),
  ('h', 21.875053, 12.857063)],
 [('f', 14.799841, 12.948208),
  ('o', 29.155346, 7.6161637),
  ('r', 12.167999, 9.165647)],
 [('o', 28.061592, 7.752

In [3]:
with open('unigram.dict', 'rb') as unigramModelFile:
    unigramModel = pickle.load(unigramModelFile)
unigramModelFile.close()

keyboard_raw = pd.read_csv("keyboard.csv")
keyboard = keyboard_raw[['key', 'x_mm', 'y_mm']]
keyboard

,key,x_mm,y_mm
0,a,4.025010,9.625024
1,b,18.900047,13.650034
2,c,12.950032,13.650034
3,d,9.975025,9.625024
4,e,8.487521,5.600014
5,f,12.950032,9.625024
6,g,15.925039,9.625024
7,h,18.900047,9.625024
8,i,23.362558,5.600014
9,j,21.875053,9.625024


In [6]:
with open('unigram.dict', 'rb') as f:
    unigram_dict = pickle.load(f)

unigram_dict

{'the': np.float64(0.03933837507090547),
 'of': np.float64(0.022362525338300483),
 'and': np.float64(0.022100157619537028),
 'to': np.float64(0.020636764209678228),
 'a': np.float64(0.015440912627459126),
 'in': np.float64(0.014400707674149294),
 'for': np.float64(0.010088551882990708),
 'is': np.float64(0.008001275333472512),
 'on': np.float64(0.006376923565241907),
 'that': np.float64(0.005781144503654221),
 'by': np.float64(0.005696158661744654),
 'this': np.float64(0.005489435157037871),
 'with': np.float64(0.0054123101306521575),
 'i': np.float64(0.0052475738476111455),
 'you': np.float64(0.005094469709217781),
 'it': np.float64(0.004783281792247097),
 'not': np.float64(0.00447777365836533),
 'or': np.float64(0.004405089635955221),
 'be': np.float64(0.004078601220057391),
 'are': np.float64(0.00406991378324621),
 'from': np.float64(0.0038692427175953605),
 'at': np.float64(0.003863593258032305),
 'as': np.float64(0.0038213555280641335),
 'your': np.float64(0.0035061751858210246),


In [4]:
keyboard_dict = {}

with open('keyboard.csv', 'r') as f:
    reader = csv.reader(f)
    header = next(reader)
    col_indices = {name.strip(): i for i, name in enumerate(header)}

    for row in reader:
        try:
            key = row[col_indices['key']]
            width = float(row[col_indices['width']])
            height = float(row[col_indices['height']])
            x_mm = float(row[col_indices['x_mm']])
            y_mm = float(row[col_indices['y_mm']])

            center_x = x_mm + (width / 2.0)
            center_y = y_mm + (height / 2.0)

            keyboard_dict[key] = {
                'center': (center_x, center_y),
                'x': x_mm,
                'y': y_mm,
                'w': width,
                'h': height
            }
        except (ValueError, IndexError, KeyError):
            pass

keyboard_dict


{'a': {'center': (5.51251365, 11.637529),
  'x': 4.02501,
  'y': 9.625024,
  'w': 2.9750073,
  'h': 4.02501},
 'b': {'center': (20.38755065, 15.662538999999999),
  'x': 18.900047,
  'y': 13.650034,
  'w': 2.9750073,
  'h': 4.02501},
 'c': {'center': (14.437535650000001, 15.662538999999999),
  'x': 12.950032,
  'y': 13.650034,
  'w': 2.9750073,
  'h': 4.02501},
 'd': {'center': (11.462528650000001, 11.637529),
  'x': 9.975025,
  'y': 9.625024,
  'w': 2.9750073,
  'h': 4.02501},
 'e': {'center': (9.97502465, 7.6125187),
  'x': 8.487521,
  'y': 5.6000137,
  'w': 2.9750073,
  'h': 4.02501},
 'f': {'center': (14.437535650000001, 11.637529),
  'x': 12.950032,
  'y': 9.625024,
  'w': 2.9750073,
  'h': 4.02501},
 'g': {'center': (17.41254265, 11.637529),
  'x': 15.925039,
  'y': 9.625024,
  'w': 2.9750073,
  'h': 4.02501},
 'h': {'center': (20.38755065, 11.637529),
  'x': 18.900047,
  'y': 9.625024,
  'w': 2.9750073,
  'h': 4.02501},
 'i': {'center': (24.85006165, 7.6125187),
  'x': 23.362558,

In [7]:
def gaussian_prob(touch_point, key_center, sigma_x = 1.0, sigma_y = 1.0):

    x, y = touch_point
    mean_x, mean_y = key_center

    prob_x = stats.norm.pdf(x, loc=mean_x, scale=sigma_x)
    prob_y = stats.norm.pdf(y, loc=mean_y, scale=sigma_y)

    return prob_x * prob_y

In [8]:
def get_decoded_word(touch_points):

    n = len(touch_points)
    if n == 0:
        return ""

    touch_coords = [(tp[1], tp[2]) for tp in touch_points]

    possible_words = {word: prob for word, prob in unigram_dict.items() if len(word) == n}

    if not possible_words:
        return ""

    best_word = ""
    max_log_prob = -float('inf')

    epsilon = 1e-300

    for word, prob_w in possible_words.items():

        current_log_prob = math.log(prob_w + epsilon)

        valid_word = True
        for i in range(n):
            char = word[i]
            if char not in keyboard_dict:
                valid_word = False
                break

            touch_point = touch_coords[i]
            key_center = keyboard_dict[char]

            prob_si_ci = gaussian_prob(touch_point, key_center)

            current_log_prob += math.log(prob_si_ci + epsilon)

        if valid_word and current_log_prob > max_log_prob:
            max_log_prob = current_log_prob
            best_word = word

    return best_word

# **Unigram Language Model Decoder**

In [9]:
key_width = 3
key_height = 4
a = 2.403
b = 0.017
c = 2.295
d = 0.016

def get_likelihood(p, mu, sigma):

    x, y = p
    mean_x, mean_y = mu
    sigma_x, sigma_y = sigma

    prob_x = stats.norm.pdf(x, loc=mean_x, scale=sigma_x)
    prob_y = stats.norm.pdf(y, loc=mean_y, scale=sigma_y)

    lik = prob_x * prob_y
    return lik

def is_letter(p, letter):

    if letter not in keyboard_dict:
        return False

    key_data = keyboard_dict[letter]
    px, py = p

    x_min = key_data['x']
    y_min = key_data['y']
    x_max = key_data['x'] + key_data['w']
    y_max = key_data['y'] + key_data['h']

    return (x_min <= px < x_max) and (y_min <= py < y_max)

def get_literal_string(touch_points):

    literal_str = ""
    for tp in touch_points:
        touch_coord = (tp[1], tp[2])
        found_key = ''

        for key, data in keyboard_dict.items():
            if is_letter(touch_coord, key):
                found_key = key
                break

        if not found_key:
            min_dist = float('inf')
            closest_key = ''
            for key, data in keyboard_dict.items():
                dist = math.dist(touch_coord, data['center'])
                if dist < min_dist:
                    min_dist = dist
                    closest_key = key
            literal_str += closest_key
        else:
            literal_str += found_key

    return literal_str

In [10]:
def unigram_lm_decoder(touchpoints):

    n = len(touchpoints)
    if n == 0:
        return ""

    touch_coords = [(tp[1], tp[2]) for tp in touchpoints]

    possible_words = {word: prob for word, prob in unigram_dict.items() if len(word) == n}

    if not possible_words:
        return ""

    best_word = ""
    max_log_prob = -float('inf')

    epsilon = 1e-300

    for word, prob_w in possible_words.items():

        current_log_prob = math.log(prob_w + epsilon)

        valid_word = True
        for i in range(n):
            char = word[i]
            if char not in keyboard_dict:
                valid_word = False
                break

            key_data = keyboard_dict[char]

            key_center = key_data['center']

            sigma_x = a + b * key_data['w']
            sigma_y = c + d * key_data['h']

            touch_point = touch_coords[i]

            prob_si_ci = get_likelihood(touch_point, key_center, (sigma_x, sigma_y))

            current_log_prob += math.log(prob_si_ci + epsilon)

        if valid_word and current_log_prob > max_log_prob:
            max_log_prob = current_log_prob
            best_word = word

    return best_word

In [11]:
decoded_success_count = 0
literal_success_count = 0
decoded_words = []
literal_strings = []
correct_words = []
for touchpoints in words:

    correct_word = "".join([tp[0] for tp in touchpoints])
    correct_words.append(correct_word)

    decoded_word = unigram_lm_decoder(touchpoints)
    decoded_words.append(decoded_word)

    literal_string = get_literal_string(touchpoints)
    literal_strings.append(literal_string)

    if decoded_word == correct_word:
        decoded_success_count += 1
    if literal_string == correct_word:
        literal_success_count += 1

total_words = len(words)
if total_words > 0:
    decoded_rate = decoded_success_count / total_words
    literal_rate = literal_success_count / total_words
else:
    decoded_rate = 0.0
    literal_rate = 0.0

with open("results.txt", 'w') as output:

    output.write(f"{decoded_rate}, {literal_rate}\n")

    for i in range(total_words):
        output.write(f"{correct_words[i]}, {decoded_words[i]}, {literal_strings[i]}\n")

Final success_rate of decoded_words : 91.5%

Final success_rate of literal_strings: 28.5%